# Ryan PINN: start from the beginning
Enable a GPU runtime, then run cells in order. Everything is filled in: no SHA or path placeholders. This notebook uses its own Drive folder, `ryan_pinn_trial_from_start_0677b40`, so earlier experiment folders are preserved.

Run one training cell at a time. There are six primary training cells, primary evaluation, six stress training cells, stress evaluation, deliverable generation, and ZIP download. Each model prints training progress every 100 updates. All scientific code runs from the reviewed revision below.

If the runtime disconnects, reopen this same notebook and repeat setup. It reuses this folder and skips completed models. Enable the recovery setting only after a training process was interrupted. Use one notebook/runtime on this folder at a time.


In [ ]:
COMMIT_SHA = "0677b40869fabf50035075e240aea3b98a0c7e05"
RECOVER_INFRASTRUCTURE = False
from google.colab import drive
from pathlib import Path
import re
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT_SHA)
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/ryan_pinn_trial_from_start_0677b40")
REPO = BASE / "repository"
print("Experiment SHA:", COMMIT_SHA)
print("New experiment folder:", BASE)


In [ ]:
import subprocess
import sys
import json
import shutil
from datetime import datetime, timezone
assert Path("/content/drive/MyDrive").is_dir(), "Mount Drive first"
BASE.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Arawal123/helmholtz-pinn-rff-benchmark.git", str(REPO)], check=True)
assert (REPO / ".git").is_dir(), f"Inspect {REPO}: it is not a Git checkout"

def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO, text=True).strip()
assert not git("status", "--porcelain"), "Checkout has source edits; preserve and inspect them"
attempts = sorted(p for p in (REPO / "results/raw").glob("n*/**/attempt_*") if p.is_dir())
if attempts:
    assert git("rev-parse", "HEAD") == COMMIT_SHA, "Keep existing attempts at their original revision"
    for attempt in attempts:
        record = attempt / "run.json"
        if record.exists():
            state = json.loads(record.read_text(encoding="utf-8"))
            assert state.get("provenance", {}).get("git_sha") in (None, COMMIT_SHA), "Mixed experiment revisions"
subprocess.run(["git", "fetch", "origin"], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=REPO, check=True)
assert git("rev-parse", "HEAD") == COMMIT_SHA
assert not git("status", "--porcelain")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO, check=True)
if not (BASE / "EXPERIMENT_LOG_execution.md").exists():
    shutil.copy2(REPO / "EXPERIMENT_LOG.md", BASE / "EXPERIMENT_LOG_execution.md")
    with (BASE / "EXPERIMENT_LOG_execution.md").open("a", encoding="utf-8") as journal:
        journal.write(f"\n{datetime.now(timezone.utc).isoformat()}: User requested a fresh run from the beginning. "
                      f"Independent folder={BASE}; frozen SHA={COMMIT_SHA}. Earlier experiment outputs remain "
                      "in /content/drive/MyDrive/ryan_pinn_trial and are not combined with this run.\n")
print("Setup ready. Existing training attempts in THIS folder:", len(attempts))
print("Run the validation cell next.")


In [ ]:
from datetime import datetime, timezone
import shutil
import signal

def run_stage(*args):
    """Launch repository CLI and persist its console output; no scientific logic."""
    assert git("rev-parse", "HEAD") == COMMIT_SHA, "Experiment revision changed: stop and inspect"
    assert not git("status", "--porcelain"), "Experiment checkout has edits"
    if RECOVER_INFRASTRUCTURE and "--recover-infrastructure" not in args and "status" not in args:
        args = (*args, "--recover-infrastructure")
    log = BASE / ("console_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".log")
    command = [sys.executable, "-u", "scripts/run_frozen.py", "--config", "configs/frozen.yaml", *args]
    print("Command:", " ".join(command), "\nLog:", log)
    with log.open("x", encoding="utf-8") as output:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                output.write(line)
                output.flush()
            returncode = process.wait()
        except BaseException:
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
                try:
                    process.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

# A tiny, unrelated derivative check verifies the installed CUDA build can execute.
# It does not construct or train a protected benchmark model.
cuda_check = """
import torch
assert torch.cuda.is_available(), 'GPU unavailable: select a GPU runtime and repeat setup'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
x = torch.tensor([0.25], device='cuda', requires_grad=True)
y = torch.sin(x)
dy = torch.autograd.grad(y.sum(), x, create_graph=True)[0]
d2y = torch.autograd.grad(dy.sum(), x)[0]
assert torch.isfinite(d2y).all().item()
torch.cuda.synchronize()
print('CUDA derivative preflight passed')
"""
subprocess.run([sys.executable, "-c", cuda_check], cwd=REPO, check=True)
subprocess.run([sys.executable, "scripts/validate_protocol.py"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, check=True)
(BASE / "code_sha.txt").write_text(COMMIT_SHA + "\n", encoding="utf-8")
environment_log = BASE / ("environment_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".txt")
with environment_log.open("x", encoding="utf-8") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=True)
if not (BASE / "EXPERIMENT_LOG_execution.md").exists():
    shutil.copy2(REPO / "EXPERIMENT_LOG.md", BASE / "EXPERIMENT_LOG_execution.md")
print("Validation passed. Run one training cell at a time; do not start another notebook on these results.")
print("Explicit infrastructure recovery:", RECOVER_INFRASTRUCTURE)
run_stage("--action", "status")


## n=6: primary condition
Run each training cell once. Repeating a terminal slot safely skips it. Preserve numerical failures.

**Train n=6, vanilla, seed 0.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "0")

**Train n=6, vanilla, seed 1.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "1")

**Train n=6, vanilla, seed 2.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "2")

**Train n=6, rff, seed 0.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "0")

**Train n=6, rff, seed 1.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "1")

**Train n=6, rff, seed 2.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "2")

**Evaluate n=6.** The repository refuses this cell until all six training slots for this condition are terminal.

In [ ]:
run_stage("--action", "evaluate", "--n", "6")

## n=12: preregistered stress/failure condition
Run each training cell once. Repeating a terminal slot safely skips it. Preserve numerical failures.

**Train n=12, vanilla, seed 0.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "0")

**Train n=12, vanilla, seed 1.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "1")

**Train n=12, vanilla, seed 2.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "2")

**Train n=12, rff, seed 0.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "0")

**Train n=12, rff, seed 1.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "1")

**Train n=12, rff, seed 2.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "2")

**Evaluate n=12.** The repository refuses this cell until all six training slots for this condition are terminal.

In [ ]:
run_stage("--action", "evaluate", "--n", "12")

## Generate all deliverables
This cell requires all 12 training slots and both condition evaluations to be complete. It generates the five processed CSVs, figures, and methods/results note.

In [ ]:
run_stage("--action", "finalize")
run_stage("--action", "status")

## Package and download
Append observations, scientific failures, interruptions, and decisions to the external execution log. This archive contains source, frozen assets, raw/processed evidence, figures, report, SHA, environment, console logs, and journal.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = BASE / ("ryan_pinn_submission_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".zip")
excluded = {".git", "__pycache__", ".pytest_cache", "smoke_outputs"}
with ZipFile(archive, "x", ZIP_DEFLATED) as z:
    for path in BASE.rglob("*"):
        if path.is_file() and path.suffix != ".zip" and not excluded.intersection(path.relative_to(BASE).parts):
            z.write(path, path.relative_to(BASE))
print("Saved:", archive)
files.download(str(archive))

## Resume after a runtime interruption
Reopen this notebook and use the same Drive folder and experiment SHA. Ensure the previous process has stopped, set `RECOVER_INFRASTRUCTURE = True` in the first cell, repeat setup/validation, then rerun the affected training or evaluation cell. Set it back to False after recovery. Prior attempts are preserved and interrupted training restarts from update 1. Completed and scientifically failed slots are skipped. Do not start another notebook on these results.
